In [ ]:
!pip install -q torch numpy soundfile pandas jiwer transformers openai-whisper torchaudio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 20.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.5 MB

In [ ]:
import torch
import numpy as np
import soundfile as sf
import time
import pandas as pd
from typing import Dict, List
from jiwer import wer
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    AutoModelForCTC,
    AutoProcessor
)
import whisper
import torchaudio
import logging
from dataclasses import dataclass
from pathlib import Path
import json
import os

In [ ]:
import torch
import numpy as np
import soundfile as sf
import time
import pandas as pd
from typing import Dict, List
from jiwer import wer
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    AutoModelForCTC,
    AutoProcessor
)
import whisper
import torchaudio
import logging
from dataclasses import dataclass
from pathlib import Path
import json

@dataclass
class EvaluationMetrics:
    wer: float
    rtf: float
    latency: float
    cpu_memory: float
    gpu_memory: float
    inference_speed: float


class STTEvaluator:
    def __init__(self, audio_dir: str, reference_dir: str):
        self.audio_dir = Path(audio_dir)
        self.reference_dir = Path(reference_dir)
        self.results = {}
        self.setup_logging()
        self.target_sample_rate = 16000  # Standard sample rate for most models

    def setup_logging(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger('STTEvaluator')

    def load_audio(self, audio_path: str):
        """Load and preprocess audio file."""
        waveform, sample_rate = torchaudio.load(audio_path)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample if necessary
        if sample_rate != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(sample_rate, self.target_sample_rate)
            waveform = resampler(waveform)

        return waveform.squeeze().numpy(), self.target_sample_rate

    def measure_memory_usage(self):
        """Measure CPU and GPU memory usage."""
        cpu_memory = 0
        gpu_memory = 0

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gpu_memory = torch.cuda.max_memory_allocated() / 1024**2  # MB

        import psutil
        cpu_memory = psutil.Process().memory_info().rss / 1024**2  # MB

        return cpu_memory, gpu_memory

    def evaluate_whisper_hf(self, model_name: str):
        """Evaluate Whisper model using Hugging Face implementation."""
        try:
            processor = WhisperProcessor.from_pretrained(model_name)
            model = WhisperForConditionalGeneration.from_pretrained(model_name)

            if torch.cuda.is_available():
                model = model.to("cuda")

            def transcribe(audio, sr):
                inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
                if torch.cuda.is_available():
                    inputs = {k: v.to("cuda") for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model.generate(**inputs)
                return processor.batch_decode(outputs, skip_special_tokens=True)[0]

            metrics = self._evaluate_model(
                transcribe,
                f"{model_name}"
            )
            return metrics

        except Exception as e:
            self.logger.error(f"Error evaluating {model_name}: {str(e)}")
            return None

    def evaluate_wav2vec2(self, model_name: str):
        """Evaluate Wav2Vec2 model."""
        try:
            processor = Wav2Vec2Processor.from_pretrained(model_name)
            model = Wav2Vec2ForCTC.from_pretrained(model_name)

            if torch.cuda.is_available():
                model = model.to("cuda")

            def transcribe(audio, sr):
                inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
                if torch.cuda.is_available():
                    inputs = inputs.input_values.to("cuda")

                with torch.no_grad():
                    logits = model(inputs).logits
                predicted_ids = torch.argmax(logits, dim=-1)
                return processor.batch_decode(predicted_ids)[0]

            metrics = self._evaluate_model(
                transcribe,
                f"{model_name}"
            )
            return metrics

        except Exception as e:
            self.logger.error(f"Error evaluating {model_name}: {str(e)}")
            return None

    def _evaluate_model(self, transcribe_fn, model_name: str):
        """Core evaluation logic for any model."""
        total_wer = 0
        total_time = 0
        total_audio_duration = 0
        processing_times = []

        try:
            for audio_file in self.audio_dir.glob("*.wav"):
                reference_file = self.reference_dir / (audio_file.stem + ".txt")

                if not reference_file.exists():
                    self.logger.warning(f"No reference file found for {audio_file}")
                    continue

                # Load audio and reference
                audio, sr = self.load_audio(str(audio_file))
                with open(reference_file, 'r', encoding='utf-8') as f:
                    reference = f.read().strip()

                # Measure transcription time and performance
                start_time = time.time()
                hypothesis = transcribe_fn(audio, sr)
                end_time = time.time()

                processing_time = end_time - start_time
                processing_times.append(processing_time)

                # Calculate metrics
                current_wer = wer(reference, hypothesis)
                audio_duration = len(audio) / sr
                total_wer += current_wer
                total_time += processing_time
                total_audio_duration += audio_duration

            # Get memory usage
            cpu_memory, gpu_memory = self.measure_memory_usage()

            # Calculate aggregate metrics
            avg_wer = total_wer / len(processing_times)
            rtf = total_time / total_audio_duration
            avg_latency = np.mean(processing_times)

            metrics = EvaluationMetrics(
                wer=avg_wer,
                rtf=rtf,
                latency=avg_latency,
                cpu_memory=cpu_memory,
                gpu_memory=gpu_memory,
                inference_speed=1/avg_latency,

            )

            self.results[model_name] = metrics
            return metrics

        except Exception as e:
            self.logger.error(f"Error in evaluation loop for {model_name}: {str(e)}")
            return None

    def export_results(self, output_file: str):
        """Export evaluation results to JSON."""
        results_dict = {
            model_name: {
                "wer": metrics.wer,
                "rtf": metrics.rtf,
                "latency": metrics.latency,
                "cpu_memory_mb": metrics.cpu_memory,
                "gpu_memory_mb": metrics.gpu_memory,
                "inference_speed": metrics.inference_speed,

            }
            for model_name, metrics in self.results.items()
            if metrics is not None  # Only include successful evaluations
        }

        with open(output_file, 'w') as f:
            json.dump(results_dict, f, indent=4)

def main():
    # Initialize evaluator
    evaluator = STTEvaluator(
        audio_dir="/content/audio",
        reference_dir="/content/transcripts"
    )

    # List of models to evaluate
    whisper_models = [
         "openai/whisper-tiny",
        "openai/whisper-base",
        "openai/whisper-small",
        "openai/whisper-medium",
        "openai/whisper-large-v2"
    ]

    wav2vec2_models = [
        "facebook/wav2vec2-base-960h",
        "facebook/wav2vec2-large-960h",
        "addy88/wav2vec2-english-stt",
        "jonatasgrosman/wav2vec2-large-xlsr-53-english",
        "microsoft/wavlm-large"
    ]

    # Evaluate Whisper models
    for model_name in whisper_models:
        evaluator.evaluate_whisper_hf(model_name)

    # Evaluate Wav2Vec2 models
    for model_name in wav2vec2_models:
        evaluator.evaluate_wav2vec2(model_name)

    # Export results
    evaluator.export_results("stt_evaluation_results.json")

In [ ]:
if __name__ == "__main__":
    main()

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
ERROR:STTEvaluator:Error evaluating microsoft/wavlm-large: Can't load tokenizer for 'microsoft/wavlm-large'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/wavlm-large' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.
ERROR:STTEvaluator:Error evaluating nvidia/stt_en_confo

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

You are using a model of type hubert to instantiate a model of type wav2vec2. This is not supported for all configurations of models and can yield errors.


pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

ERROR:STTEvaluator:Error evaluating facebook/hubert-large-ls960-ft: The state dictionary of the model you are trying to load is corrupted. Are you sure it was properly saved?
ERROR:STTEvaluator:Error evaluating microsoft/hubert-base-ls960: microsoft/hubert-base-ls960 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
